# B1.0 · What an agentic harness is

**Function B — Application Security with an AI SDLC → The Agentic Harness**  ·  *Both directions*

Builds on **[A3.10 · The agent's escalation path](https://spbreed.github.io/cyber-commons/lessons/A3.10.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Almost everyone running agents cannot name their harness's verifier, and "the model tells us" means there isn't one. Eight components, named once — and most arguments about agent reliability turn out to be arguments about which of the eight is missing.

> **At CyberTravels.** CyberTravels already runs four harnesses and calls them agents. The Workflow Agent is a loop over booking tools whose verifier nobody ever specified — which is the answer to why it refunded twice.

## 2 · The framework

```
   the eight components of any harness

   +-----------+   +--------+   +---------+   +-----------+
   |   model   |   |  loop  |   |  tools  |   |  context  |
   +-----------+   +--------+   +---------+   +-----------+
   +-----------+   +--------+   +---------+   +-----------+
   | verifier  |   | budget |   | memory  |   |orchestrator|
   +-----------+   +--------+   +---------+   +-----------+
                                              (+ telemetry)

   the one people cannot name is almost always the verifier
   -- and when something goes wrong, the class of failure
      is just "which of these eight did it"
```

**The model is not the system.**

A model is a text generator. Give it tokens, get tokens back. It has no memory
between calls, no ability to act, and no notion of whether it succeeded. Left
alone it cannot read a file, run a scanner or open a pull request.

> **A harness is everything wrapped around a model that turns generating text
> into getting work done.** It decides what the model sees, what it is allowed
> to do, whether what it did worked, when to stop, and what is written down
> afterwards.

That is the definition, and it is worth being pedantic about, because "agent",
"scaffold", "framework" and "harness" get used interchangeably and the
substitution hides the question that matters: *which of these eight parts do
you actually have?*

| Component | What it does |
|---|---|
| **The loop** | Decides what happens next — plan, act, observe, decide again — and when to stop |
| **Tools** | The only way the model touches the world |
| **Context management** | What the model sees at each step, assembled from a world much larger than the window |
| **The verifier** | The independent check on whether a step actually succeeded |
| **State & memory** | What survives between steps and between runs |
| **Budgets & stop conditions** | Token, time, cost and action ceilings that bound autonomy |
| **Orchestrator** | Sub-agent spawning, parallelism, delegation depth |
| **Telemetry** | The record that makes a run auditable, replayable and debuggable |

**CyberTravels already runs four of these and calls them agents.** The Workflow
Agent is a loop over booking tools with a verifier nobody specified. The Coding
Agent is a loop over a repository with repository write. Chapter 5 adds a fifth
— the pipeline that reviews the Coding Agent's pull requests — and every stage
of it is another instance of this same eight-part shape.

Two teams given the **identical model** routinely differ by an order of
magnitude in output quality, purely on harness design. Most of the capability
you attribute to a model is the scaffold around it.

And the security consequence is direct, which is why this chapter sits in
Function B rather than in a tooling appendix: a harness is itself an autonomous
actor holding credentials and tools. Every risk in Function A applies to it.

## 3 · Build the smallest harness that is still a harness

Eight components, none optional. The model here is a deterministic stand-in — labelled as one — so the scaffold is what you can see.

In [ ]:
from dataclasses import dataclass, field

def stand_in_model(prompt):
    """NOT a language model. A deterministic stub, so the harness is visible.

    It reads the transcript so far to decide what is left to do - which is all
    any agent loop does, minus the part that is hard."""
    if "write_patch" not in prompt:
        return {"tool": "write_patch", "args": {"file": "auth.py"}}
    if "run_tests" not in prompt:
        return {"tool": "run_tests", "args": {}}
    return {"tool": "done", "args": {"claim": "fixed it"}}

WORLD = {"tests_pass": False, "patched": False}

def run_tests(**_):
    # the patch this stub writes does not actually fix the bug
    return {"passed": WORLD["tests_pass"], "failing": [] if WORLD["tests_pass"] else ["test_login"]}
def write_patch(file, **_):
    WORLD["patched"] = True
    return {"wrote": file}
def done(claim, **_):
    return {"claim": claim}

TOOLS = {"run_tests": run_tests, "write_patch": write_patch, "done": done}

@dataclass
class Budget:
    steps: int = 6
    used: int = 0
    def spend(self):
        self.used += 1
        return self.used <= self.steps

def harness(task, verifier=None, budget=None, telemetry=None):
    """loop + tools + context + verifier + state + budget + telemetry."""
    budget = budget or Budget()
    telemetry = telemetry if telemetry is not None else []
    context = [f"TASK: {task}"]                       # context management
    state = {"steps": 0}                              # state
    while budget.spend():                             # budgets / stop conditions
        step = stand_in_model("\n".join(context))     # the model
        tool, args = step["tool"], step["args"]
        result = TOOLS[tool](**args)                  # tools
        state["steps"] += 1
        telemetry.append({"step": state["steps"], "tool": tool, "result": result})
        context.append(f"{tool} -> {result}")
        if tool == "done":
            ok = verifier() if verifier else True     # the verifier
            return {"claimed": True, "verified": ok, "steps": state["steps"],
                    "telemetry": telemetry}
    return {"claimed": False, "verified": False, "steps": state["steps"],
            "telemetry": telemetry}

print("components wired:", ["loop","tools","context","verifier","state",
                            "budget","orchestrator","telemetry"])

## 4 · Run it once with no verifier

In [ ]:
WORLD.update(tests_pass=False, patched=False)
r = harness("fix the failing test_login", verifier=None)
print(f"agent claimed success : {r['claimed']}")
print(f"independently checked : {r['verified']}")
print(f"steps                 : {r['steps']}")
for t in r["telemetry"]:
    print(f"   {t['step']}. {t['tool']:12s}{t['result']}")
print()
print("It reported success. The tests still fail. Nothing in that transcript")
print("is a lie - the agent did write a patch, and then it said it was done.")
assert r["claimed"] and not WORLD["tests_pass"]

## 5 · Where it breaks — the component people leave out

Add the verifier and change nothing else.

In [ ]:
def real_verifier():
    """Ground truth, not self-assessment: run the tests and read the result."""
    return run_tests()["passed"]

WORLD.update(tests_pass=False, patched=False)
r2 = harness("fix the failing test_login", verifier=real_verifier)
print(f"claimed {r2['claimed']}  verified {r2['verified']}")

WORLD.update(tests_pass=True)          # now the fix actually works
r3 = harness("fix the failing test_login", verifier=real_verifier)
print(f"claimed {r3['claimed']}  verified {r3['verified']}")
print()
print("Same model. Same loop. Same tools. The only difference between a harness")
print("that reports the truth and one that reports its own optimism is one")
print("component - and it is the cheapest one in the table.")
assert not r2["verified"] and r3["verified"]

## 6 · The budget is a security control, not a cost control

In [ ]:
def looping_model(prompt):
    return {"tool": "run_tests", "args": {}}          # never finishes

import builtins
_orig = stand_in_model
try:
    globals()["stand_in_model"] = looping_model
    WORLD.update(tests_pass=False)
    r4 = harness("fix it", verifier=real_verifier, budget=Budget(steps=4))
finally:
    globals()["stand_in_model"] = _orig

print(f"ran {r4['steps']} steps, then stopped: claimed={r4['claimed']}")
print()
print("Without the ceiling this runs until something else stops it - a bill, a")
print("rate limit, or an on-call engineer. The budget is what makes 'autonomous'")
print("a bounded word.")
assert r4["steps"] == 4 and not r4["claimed"]

## 7 · Verify — the harness is itself an actor

It holds credentials and calls tools. Score it the way you would score any other non-human identity.

In [ ]:
HARNESS_ACTOR = {
 "identity": "ci-sast-harness",
 "tools": sorted(TOOLS),
 "writes": ["write_patch"],
 "credentials": ["repo:write"],
 "runs_unattended": True,
 "telemetry": True,
}
irreversible = [t for t in HARNESS_ACTOR["writes"]]
print(f"{'property':22s}value")
for k, v in HARNESS_ACTOR.items():
    print(f"{k:22s}{v}")
print()
print(f"tools that change state : {irreversible}")
print(f"unattended              : {HARNESS_ACTOR['runs_unattended']}")
print(f"auditable               : {HARNESS_ACTOR['telemetry']}")
print()
print("Every question you would ask of an agent applies to the thing you just")
print("built to review agents. A harness with repo:write running unattended is")
print("a non-human identity, and it belongs in the inventory in A2 and E1.2.")
assert HARNESS_ACTOR["telemetry"], "an unauditable harness cannot be governed"

## 8 · When it goes wrong, which of the eight failed?

"The agent messed up" is not a defect report: it routes to nobody, and every incident feels novel. Naming the part that failed turns an incident into a ticket with an owner.

In [ ]:
INCIDENTS = [
 ("agent's patch did not compile; the loop retried and fixed it", None),
 ("agent's patch passed CI and introduced a SQL injection", None),
 ("agent deleted a production table it should never have had access to", None),
 ("agent posted the contents of .env to a public issue", None),
 ("agent approved a PR because a code comment told it to", None),
 ("agent looped for 6 hours re-running the same failing test", None),
 ("agent opened the same pull request 14 times", None),
 ("agent could not solve the task and correctly reported failure", None),
]
TAXONOMY = {
 "capability":   ("the model could not do it",              "better model / better context"),
 "verification": ("it did it wrong and we believed it",     "harness engineer — B1.1"),
 "authority":    ("it did what it should not be able to do","identity — A2"),
 "containment":  ("the action reached further than intended","platform — A3"),
 "injection":    ("untrusted content drove it",             "provenance — A2.6"),
 "budget":       ("it never stopped",                       "harness engineer — A3.4"),
 "idempotency":  ("it did the right thing twice",           "harness engineer — B1.2"),
}
LABELS = ["capability", "verification", "authority", "containment",
          "injection", "budget", "idempotency", "capability"]

for (text, _), label in zip(INCIDENTS, LABELS):
    what, owner = TAXONOMY[label]
    print(f"{label:13s} {text}")
    print(f"{'':13s} → {owner}")

## 9 · Where it breaks — the two that get confused

Incidents 1 and 2 both start "the agent's patch was wrong". They are different defects with different owners, and conflating them is how a team spends a quarter upgrading models to fix a verifier.

In [ ]:
def classify(produced_wrong_output, harness_accepted_it, action_taken):
    """The decision rule that separates capability from verification."""
    if not produced_wrong_output:
        return "not a model failure"
    if not harness_accepted_it:
        return "capability — the harness caught it, the loop worked"
    if action_taken:
        return "VERIFICATION — the harness shipped wrong work"
    return "verification (contained) — accepted but nothing acted on it"

CASES = [
 ("patch did not compile, loop retried",  True,  False, False),
 ("patch passed CI, shipped SQLi",        True,  True,  True),
 ("patch wrong, accepted, never merged",  True,  True,  False),
 ("patch correct",                        False, True,  True),
]
for name, wrong, accepted, acted in CASES:
    print(f"{name:38s} → {classify(wrong, accepted, acted)}")
print("\nThe model was equally wrong in the first three. Only one is YOUR defect.")

## 10 · The control — classify automatically from the trace

The taxonomy is only useful if applying it is cheap. Most of the classification is derivable from what the harness already records.

In [ ]:
def classify_from_trace(trace):
    """trace: dict of facts the harness already has."""
    if trace.get("denied_by_policy"):        return "authority"
    if trace.get("denied_by_sandbox"):       return "containment"
    if trace.get("instruction_source") not in (None, "principal"):
        return "injection"
    if trace.get("stopped_by", "").startswith(("step budget", "time budget")):
        return "budget"
    if trace.get("duplicate_effect"):        return "idempotency"
    if trace.get("verifier_passed") and trace.get("outcome_wrong"):
        return "verification"
    if trace.get("outcome_wrong"):           return "capability"
    return "success"

TRACES = [
 {"verifier_passed": False, "outcome_wrong": True, "stopped_by": "step budget (5 steps)"},
 {"verifier_passed": True,  "outcome_wrong": True},
 {"denied_by_policy": True},
 {"denied_by_sandbox": True},
 {"instruction_source": "pull-request-diff"},
 {"stopped_by": "time budget (300s)"},
 {"duplicate_effect": True},
 {"verifier_passed": True,  "outcome_wrong": False},
]
for t in TRACES:
    cls = classify_from_trace(t)
    owner = TAXONOMY.get(cls, ("", "—"))[1]
    print(f"{cls:14s} {owner:32s} {t}")

counts = {}
for t in TRACES:
    c = classify_from_trace(t); counts[c] = counts.get(c, 0) + 1
print(f"\ndistribution: {counts}")
assert classify_from_trace(TRACES[1]) == "verification"

## What you just proved

The smallest harness that is still a harness runs its loop, and then the same loop with a verifier added refuses the work it previously accepted. The budget stops a looping model. The harness's own identity, scopes and logging show it is an actor like any other. Eight incidents then classify into seven failure classes, and the two that look identical from outside — capability and verification — separate on one rule: did the harness accept it.

## Your turn

Take your last agent incident and name which of the eight components failed. If the answer is "the model", check whether the harness accepted the output — if it did, the defect is yours, not the model's.

---

**Next → [B1.1 · The loop, and the verifier that decides what it may conclude](https://spbreed.github.io/cyber-commons/lessons/B1.1.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.0.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.0.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*